# 12 Translate — Manipuri (Colab)

> **Before running:** `Runtime > Change runtime type > T4 GPU`

Translates the dataset into **Manipuri** using `sarvamai/sarvam-translate` on Colab GPU.

- Resumable: re-run from **Cell 8** if the session times out — checkpoint is saved to Drive.
- Output: `Translation/datasets/manipuri/fakehealth_healthfact_manipuri.csv` in your Drive.


## 1. Check GPU


In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name :', torch.cuda.get_device_name(0))
    print('VRAM     :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU and re-run.')


## 2. Install Packages


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'transformers','accelerate','sentencepiece','-q'])
print('Packages ready.')


## 3. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


## 4. Paths

> **Edit these two paths** to match where your files are in Google Drive.


In [ ]:
from pathlib import Path
import json, datetime
import pandas as pd
from tqdm.auto import tqdm

# ── EDIT THESE ────────────────────────────────────────────────────────
DATASET_PATH = Path('/content/drive/MyDrive/Reseach/Dataset/dataset/processed/fakehealth_healthfact_binary_clean.csv')
TRANS_ROOT   = Path('/content/drive/MyDrive/Reseach/Translation')
# ──────────────────────────────────────────────────────────────────────

LANGUAGE_NAME  = 'Manipuri'
LANG_STR       = 'Manipuri'
LANG_KEY       = 'manipuri'
TITLE_COL      = 'title_mni'
TEXT_COL       = 'text_mni'

LANG_DIR       = TRANS_ROOT / 'datasets' / 'manipuri'
CHECKPOINT_DIR = LANG_DIR / 'checkpoints'
FINAL_CSV      = LANG_DIR / 'fakehealth_healthfact_manipuri.csv'
CHECKPOINT_EVERY = 100

LANG_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Language : {LANGUAGE_NAME}')
print(f'Output   : {FINAL_CSV}')
print(f'Dataset exists: {DATASET_PATH.exists()}')


## 5. Load Model


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'sarvamai/sarvam-translate'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_NAME} on {DEVICE} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE)
model.eval()
print('Model ready.')


## 6. Translation Helpers


In [ ]:
def split_sentences(text: str, max_sent: int = 5) -> list:
    parts = [s.strip() for s in text.replace('. ', '.|||').split('|||') if s.strip()]
    return [' '.join(parts[i:i+max_sent]) for i in range(0,len(parts),max_sent)] or [text]


def translate_chunk(chunk: str, tgt_lang: str) -> str:
    messages = [
        {'role':'system','content':f'Translate the text below to {tgt_lang}.'},
        {'role':'user',  'content': chunk},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=True, temperature=0.01, num_return_sequences=1,
        )
    out_ids = gen_ids[0][len(inputs.input_ids[0]):].tolist()
    return tokenizer.decode(out_ids, skip_special_tokens=True).strip()


def translate_text(text: str, tgt_lang: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    return ' '.join(translate_chunk(c, tgt_lang) for c in split_sentences(text))


print('Helpers ready.')


## 7. Load Dataset and Resume


In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Dataset: {df.shape}')

ckpt_files = sorted(CHECKPOINT_DIR.glob('checkpoint_*.csv'))
if ckpt_files:
    done_df  = pd.read_csv(ckpt_files[-1])
    done_ids = set(done_df['record_id'])
    print(f'Resuming from checkpoint: {ckpt_files[-1].name}')
    print(f'Already done : {len(done_ids):,} / {len(df):,} rows')
else:
    done_df  = pd.DataFrame()
    done_ids = set()
    print(f'Starting fresh — {len(df):,} rows to translate')


## 8. Translation Loop

> If the Colab session times out, re-run Cells 1–7 then re-run this cell — it will resume from the last checkpoint automatically.


In [ ]:
import time
results         = done_df.to_dict('records') if not done_df.empty else []
rows_since_ckpt = 0
pending         = df[~df['record_id'].isin(done_ids)].reset_index(drop=True)
print(f'Translating {len(pending):,} rows into {LANGUAGE_NAME}...\n')
t_start = time.time()

for _, row in tqdm(pending.iterrows(), total=len(pending), desc=LANGUAGE_NAME):
    title_tr = translate_chunk(str(row['title']), LANG_STR)
    text_tr  = translate_text(str(row['text']),   LANG_STR)

    record = row.to_dict()
    record[TITLE_COL] = title_tr
    record[TEXT_COL]  = text_tr
    results.append(record)
    rows_since_ckpt += 1

    if rows_since_ckpt >= CHECKPOINT_EVERY:
        ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        pd.DataFrame(results).to_csv(
            CHECKPOINT_DIR / f'checkpoint_{ts}.csv', index=False)
        rows_since_ckpt = 0
        elapsed = round(time.time()-t_start)
        print(f'  Checkpoint saved — {len(results):,} rows done  ({elapsed}s elapsed)')

elapsed = round(time.time()-t_start,1)
print(f'\nDone. {len(results):,} rows  |  {elapsed}s ({elapsed/3600:.2f}h)')


## 9. Save Final Dataset


In [ ]:
final_df  = pd.DataFrame(results)
orig_cols = list(df.columns)
final_df  = final_df[orig_cols + [TITLE_COL, TEXT_COL]]
final_df.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')
print(f'Saved  : {FINAL_CSV}')
print(f'Shape  : {final_df.shape}')
final_df[['record_id','label_binary','title',TITLE_COL,'text',TEXT_COL]].head(3)


## 10. QA Check


In [ ]:
empty_t = final_df[final_df[TITLE_COL].str.strip().eq('')]
empty_x = final_df[final_df[TEXT_COL].str.strip().eq('')]
print('='*50)
print('QA — Manipuri')
print('='*50)
print(f'Total rows   : {len(final_df):,}')
print(f'Empty titles : {len(empty_t)}')
print(f'Empty texts  : {len(empty_x)}')
print(f'Label dist   : {final_df["label_binary"].value_counts().to_dict()}')
print('='*50)
